### 1. Persiapan dan Memuat Model Tahap 1

Pada tahap ini, kita tidak menggunakan model dasar Llama. Kita akan **melanjutkan pembelajaran** dari model `lex01_lora` yang sudah kita latih dengan kamus dan glosarium hukum pada tahap sebelumnya.

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@nightly git+https://github.com/unslothai/unsloth-zoo.git

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Memuat model hasil fine-tuning Tahap 1
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "nxvay/lex01_lora",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

Unsloth 2026.3.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


### 2. Konfigurasi LoRA Adapters Baru

Kita kembali memasang adapter LoRA untuk proses *training* ini agar hanya meng-update sebagian kecil parameter, membuat proses komputasi tetap ringan.

In [ ]:
import logging
import warnings

# Matikan peringatan dari library transformers & unsloth
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("unsloth").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 3. Dataset & Standardisasi Dataset Alpaca (ShareGPT)

Sama seperti sebelumnya, kita siapkan fungsi untuk memetakan format dataset menjadi *chat template* yang dipahami oleh Llama 3.1.

Kita menggunakan dataset instruksi bahasa Indonesia `FreedomIntelligence/alpaca-gpt4-indonesian`. Karena ukurannya sangat besar, kita **mengambil 1% sampel acak** agar proses *training* lebih cepat.

Dataset ini menggunakan format `ShareGPT` (`from`/`value`), sehingga kita harus menggunakan alat pembantu `standardize_sharegpt` dari Unsloth untuk merapikannya ke format standar Hugging Face (`role`/`content`).

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Muat dataset dan ambil 1% sampel secara acak
dataset = load_dataset("FreedomIntelligence/alpaca-gpt4-indonesian", split="train")
dataset = dataset.train_test_split(train_size = 0.01)["train"]

Repo card metadata block was not found. Setting CardData to empty.


Standardisasi format ShareGPT dan terapkan template chat

In [ ]:
from unsloth.chat_templates import standardize_sharegpt

dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

### 5. Pengecekan Hasil Format

Mari kita intip percakapan pada data ke-5 untuk memastikan format dan *system prompt* bawaan telah diterapkan dengan benar.

In [ ]:
# Melihat format asal (Role & Content)
dataset[5]["conversations"]

[{'content': 'Berikan pendapat tentang internet.\n', 'role': 'user'},
 {'content': 'Menurut pendapat saya, internet merupakan salah satu penemuan terbesar dalam sejarah manusia. Ia telah merevolusi cara kita berkomunikasi, bekerja, dan mengakses informasi. Internet membuka peluang besar bagi orang-orang, memungkinkan mereka untuk terhubung dengan orang lain, belajar keahlian baru, dan menjelajahi berbagai budaya, terlepas dari lokasi geografis mereka. Namun, pada saat yang sama, internet juga menimbulkan tantangan tersendiri, seperti masalah privasi dan keamanan online, serta penyebaran informasi yang salah. Meskipun demikian, jika digunakan dengan bertanggung jawab, internet dapat menjadi alat yang sangat kuat untuk pertumbuhan pribadi dan sosial.',
  'role': 'assistant'}]

In [ ]:
# Melihat hasil akhir teks yang akan dibaca oleh AI
dataset[5]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBerikan pendapat tentang internet.\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nMenurut pendapat saya, internet merupakan salah satu penemuan terbesar dalam sejarah manusia. Ia telah merevolusi cara kita berkomunikasi, bekerja, dan mengakses informasi. Internet membuka peluang besar bagi orang-orang, memungkinkan mereka untuk terhubung dengan orang lain, belajar keahlian baru, dan menjelajahi berbagai budaya, terlepas dari lokasi geografis mereka. Namun, pada saat yang sama, internet juga menimbulkan tantangan tersendiri, seperti masalah privasi dan keamanan online, serta penyebaran informasi yang salah. Meskipun demikian, jika digunakan dengan bertanggung jawab, internet dapat menjadi alat yang sangat kuat untuk pertumbuhan pribadi dan sosial.<|eot_id|>'

### 6. Konfigurasi SFTTrainer

Kita atur parameter *training*. Kali ini kita hanya akan menyimpan *checkpoint* (`save_steps`) setiap 20 langkah dan melakukan pelatihan penuh selama **3 epoch**.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,

        save_strategy = "steps",
        save_steps = 20,

        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/499 [00:00<?, ? examples/s]

### 7. Melatih Khusus pada Jawaban Asisten (Response Masking)

Agar fokus, model diinstruksikan untuk mengabaikan teks pertanyaan dari pengguna dan **hanya menghitung skor evaluasi (loss) dari jawabannya sendiri**.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/499 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/499 [00:00<?, ? examples/s]

Pembuktian bahwa pertanyaan pengguna berhasil disembunyikan (-100 pada label):

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBerikan pendapat tentang internet.\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nMenurut pendapat saya, internet merupakan salah satu penemuan terbesar dalam sejarah manusia. Ia telah merevolusi cara kita berkomunikasi, bekerja, dan mengakses informasi. Internet membuka peluang besar bagi orang-orang, memungkinkan mereka untuk terhubung dengan orang lain, belajar keahlian baru, dan menjelajahi berbagai budaya, terlepas dari lokasi geografis mereka. Namun, pada saat yang sama, internet juga menimbulkan tantangan tersendiri, seperti masalah privasi dan keamanan online, serta penyebaran informasi yang salah. Meskipun demikian, jika digunakan dengan bertanggung jawab, internet dapat menjadi alat yang sangat kuat untuk pertumbuhan pribadi dan sosial.<|eot_id|>'

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                           Menurut pendapat saya, internet merupakan salah satu penemuan terbesar dalam sejarah manusia. Ia telah merevolusi cara kita berkomunikasi, bekerja, dan mengakses informasi. Internet membuka peluang besar bagi orang-orang, memungkinkan mereka untuk terhubung dengan orang lain, belajar keahlian baru, dan menjelajahi berbagai budaya, terlepas dari lokasi geografis mereka. Namun, pada saat yang sama, internet juga menimbulkan tantangan tersendiri, seperti masalah privasi dan keamanan online, serta penyebaran informasi yang salah. Meskipun demikian, jika digunakan dengan bertanggung jawab, internet dapat menjadi alat yang sangat kuat untuk pertumbuhan pribadi dan sosial.<|eot_id|>'

### 8. Mulai Proses Fine-tuning Tahap 2!

Jalankan prosesnya. Kita bisa melihat nilai *loss* akan terus menurun hingga akhir Epoch 3.

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
4.211 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

{'loss': '1.506', 'grad_norm': '0.6598', 'learning_rate': '0', 'epoch': '0.016'}
{'loss': '1.502', 'grad_norm': '0.5483', 'learning_rate': '4e-05', 'epoch': '0.032'}
{'loss': '1.601', 'grad_norm': '0.7788', 'learning_rate': '8e-05', 'epoch': '0.048'}
{'loss': '1.508', 'grad_norm': '0.714', 'learning_rate': '0.00012', 'epoch': '0.064'}
{'loss': '1.419', 'grad_norm': '0.5278', 'learning_rate': '0.00016', 'epoch': '0.08'}
{'loss': '1.229', 'grad_norm': '0.7484', 'learning_rate': '0.0002', 'epoch': '0.096'}
{'loss': '0.9843', 'grad_norm': '0.6304', 'learning_rate': '0.0001989', 'epoch': '0.112'}
{'loss': '1.186', 'grad_norm': '0.7969', 'learning_rate': '0.0001978', 'epoch': '0.128'}
{'loss': '1.409', 'grad_norm': '0.7194', 'learning_rate': '0.0001967', 'epoch': '0.144'}
{'loss': '1.314', 'grad_norm': '0.6732', 'learning_rate': '0.0001957', 'epoch': '0.16'}
{'loss': '1.197', 'grad_norm': '0.9232', 'learning_rate': '0.0001946', 'epoch': '0.176'}
{'loss': '1.368', 'grad_norm': '0.6562', 'lear

In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

252.7958 seconds used for training.
4.21 minutes used for training.
Peak reserved memory = 4.221 GB.
Peak reserved memory for training = 0.01 GB.
Peak reserved memory % of max memory = 28.984 %.
Peak reserved memory for training % of max memory = 0.069 %.


### 9. Pengujian Model (Inference)

Mari kita uji kemampuan model dalam mengikuti instruksi logika sederhana menggunakan mode inferensi cepat.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Aktifkan inferensi 2x lebih cepat

messages = [
    {"role": "user", "content": "Lanjutkan bilangan fibonaci ini: 1, 1, 2, 3, 5, 8,"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 64,
    use_cache = True,
    temperature = 1.5,
    min_p = 0.1
)
print(tokenizer.batch_decode(outputs))

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContinue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe given sequence appears to be a Fibonacci sequence where each number is the sum of the two preceding numbers. The Fibonacci sequence is often given by the formula: F(n) = F(n-1) + F(n-2) for n >= 3. In this case, the Fibonacci sequence is 1, ']

### 10. Pengujian dengan TextStreamer (Efek Mengetik)

Gunakan `TextStreamer` untuk melihat jawaban model muncul per karakter secara *real-time*.

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Lanjutkan bilangan fibonaci ini: 1, 1, 2, 3, 5, 8,"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 1.5,
    min_p = 0.1
)

Bilangan fibonaci ini adalah persamaan yang bermanja:
- 1 + 1 = 2
- 1 + 1 + 2 = 4
- 1 + 1 + 2 + 3 = 9
- 1 + 1 + 2 + 3 + 5 = 15
- 1 + 1 + 2 + 3 + 5 + 8 = 20

Secara kunci, bilangan fibonaci ini dibuat menggunakan akar akar panjang dari dua bilangan fibonaci sebelumnya, dan


### 11. Menyimpan Model Tahap 2

Setelah berhasil diajari cara mengikuti instruksi, kita simpan bobot/adapter LoRA tahap 2 ini secara lokal (`lex02_lora`) dan ke Hugging Face Hub untuk dilanjutkan pada tahap final nanti.

In [ ]:
# Simpan ke memori lokal
model.save_pretrained("lex02_lora") # Local saving
tokenizer.save_pretrained("lex02_lora")

# Unggah ke repositori online Hugging Face
model.push_to_hub("nxvay/lex02_lora", token = "") # Online saving
tokenizer.push_to_hub("nxvay/lex02_lora", token = "") # Online saving

README.md:   0%|          | 0.00/579 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 23.3kB / 45.1MB            

Saved model to https://huggingface.co/nxvay/lex02_lora


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mps_vemvp3/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            